# Getting the cell potential
We seek to get the cell potential of different setups.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Arc, FancyArrowPatch, PathPatch
from matplotlib.path import Path
import numpy as np
from dataclasses import dataclass


# Library

@dataclass(frozen=True)
class Electrode:
    symbol: str
    color: str


@dataclass(frozen=True)
class Electrolyte:
    name: str
    formula: str
    charge: int


@dataclass(frozen=True)
class SaltBridge:
    name: str
    formula_display: str


SALT_BRIDGE_LIBRARY = {
    "KCl": SaltBridge("potassium chloride", "KCl"),
    "Na_2SO_4": SaltBridge("sodium sulfate", r"Na$_2$SO$_4$"),
}

ELECTRODE_LIBRARY = {
    "Zn": Electrode("Zn", "gray"),
    "Cu": Electrode("Cu", "#f4b183"),
    "Ag": Electrode("Ag", "silver"),
    "Fe": Electrode("Fe", "dimgray"),
    "Al": Electrode("Al", "lightsteelblue"),
    "Ni": Electrode("Ni", "seagreen"),
}

ELECTROLYTE_LIBRARY = {
    "chloride": Electrolyte("chloride", "Cl", -1),
    "sulfate": Electrolyte("sulfate", "SO_4", -2),
    "nitrate": Electrolyte("nitrate", "NO_3", -1),
    "chromate": Electrolyte("chromate", "CrO_4", -2),
    "hexacyanide": Electrolyte("hexacyanide", "(CN)_6", -4),
}

# Common oxidation states for demo purposes
METAL_CHARGES = {
    "Zn": 2,
    "Cu": 2,
    "Ag": 1,
    "Fe": 2,
    "Al": 3,
    "Ni": 2,
}

STANDARD_REDUCTION_POTENTIALS = {
    "Zn": -0.76,
    "Cu":  0.34,
    "Ag":  0.80,
    "Fe": -0.44,
    "Al": -1.66,
    "Ni": -0.25,
}

# Chemistry

def plain_salt_formula(metal_symbol, metal_charge, electrolyte):
    from math import gcd

    c = metal_charge
    a = abs(electrolyte.charge)

    n_metal = a
    n_anion = c
    g = gcd(n_metal, n_anion)
    n_metal //= g
    n_anion //= g

    metal_part = metal_symbol if n_metal == 1 else f"{metal_symbol}_{n_metal}"
    anion_part = format_formula_piece(electrolyte.formula, n_anion)

    return f"{metal_part}{anion_part}"

def make_cell_designation(anode_symbol, anode_charge, anode_electrolyte,
                          cathode_symbol, cathode_charge, cathode_electrolyte):
    anode_sol = plain_salt_formula(anode_symbol, anode_charge, anode_electrolyte)
    cathode_sol = plain_salt_formula(cathode_symbol, cathode_charge, cathode_electrolyte)
    return rf"$\mathbf{{Cell:}}$ $\mathrm{{{anode_symbol}\;|\;{anode_sol}\;||\;{cathode_sol}\;|\;{cathode_symbol}}}$"

def format_formula_piece(formula, n):
    if n == 1:
        return formula
    if len(formula) > 2 or "_" in formula or "(" in formula:
        return f"({formula})_{n}"
    return f"{formula}_{n}"

def build_salt_formula(metal_symbol, metal_charge, electrolyte):
    from math import gcd

    c = metal_charge
    a = abs(electrolyte.charge)

    n_metal = a
    n_anion = c
    g = gcd(n_metal, n_anion)
    n_metal //= g
    n_anion //= g

    metal_part = metal_symbol if n_metal == 1 else f"{metal_symbol}_{n_metal}"
    anion_part = format_formula_piece(electrolyte.formula, n_anion)

    return rf"$\mathbf{{{metal_part}{anion_part}}}$"

def oxidation_state_label(metal_symbol, charge):
    q = format_charge_superscript(charge)
    return rf"{metal_symbol}$^{{{q}}}$"

def make_anode_reaction(metal_symbol, charge):
    q = format_charge_superscript(charge)
    return rf"$\rightarrow$ {metal_symbol}$^{{{q}}}$"

def make_cathode_reaction(metal_symbol, charge):
    q = format_charge_superscript(charge)
    return rf"{metal_symbol}$^{{{q}}}$ $\rightarrow$"


# Draw


def draw_galvanic_cell(
    anode_metal="Zn",
    anode_electrolyte=r"$\mathbf{ZnSO_4}$",
    cathode_metal="Cu",
    cathode_electrolyte=r"$\mathbf{CuSO_4}$",
    salt_bridge_compound="KCl",
    anode_reaction=r"$\rightarrow$ Zn$^{2+}$",
    cathode_reaction=r"Cu$^{2+}$ $\rightarrow$",
    anode_half_reaction=None,
    cathode_half_reaction=None,
    cell_potential=None,
    cell_designation=None,
    title_voltmeter="Voltmeter",
    bridge_label="Salt bridge",
    anode_color="gray",
    cathode_color="#f4b183"
):
    fig, ax = plt.subplots(figsize=(20, 7), constrained_layout=True)
    ax.set_xlim(-1.2, 15.9)
    ax.set_ylim(-1.0, 9.2)
    ax.set_aspect("equal")
    ax.axis("off")

    def beaker_path(x0, y0, x1, y1):
        verts = [(x0, y1), (x0, y0), (x1, y0), (x1, y1)]
        codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.LINETO]
        return Path(verts, codes)

    def add_solution(ax, x0, x1, y_top, color, alpha):
        rect = Rectangle((x0, 0), x1 - x0, y_top, facecolor=color, edgecolor="none", alpha=alpha)
        ax.add_patch(rect)

    def add_beaker(ax, x0, y0, x1, y1, lw=2):
        path = beaker_path(x0, y0, x1, y1)
        patch = PathPatch(path, fill=False, lw=lw, joinstyle="round", capstyle="round")
        ax.add_patch(patch)

    def add_voltmeter(ax, center=(4.5, 7), radius=0.5):
        cx, cy = center
        ax.add_patch(Circle((cx, cy), radius, facecolor="white", edgecolor="black", lw=2))
        for a in range(30, 151, 30):
            th = np.deg2rad(a)
            x0 = cx + 0.35 * np.cos(th)
            y0 = cy + 0.35 * np.sin(th)
            x1 = cx + 0.45 * np.cos(th)
            y1 = cy + 0.45 * np.sin(th)
            ax.plot([x0, x1], [y0, y1], lw=1, color="tab:blue")
        ax.add_patch(Circle((cx, cy), 0.03, color="tab:blue"))
        th = np.deg2rad(60)
        x1 = cx + 0.4 * np.cos(th)
        y1 = cy + 0.4 * np.sin(th)
        ax.add_patch(FancyArrowPatch((cx, cy), (x1, y1),
                                     arrowstyle="-|>", mutation_scale=12,
                                     lw=2, color="tab:blue"))

    def plus_minus_sign(ax):
        ax.add_patch(Circle((-0.5, 5.5), 0.25, fill=False, edgecolor="red", lw=2))
        ax.plot([-0.7, -0.3], [5.5, 5.5], color="red", lw=2)
        ax.add_patch(Circle((9.5, 5.5), 0.25, fill=False, edgecolor="red", lw=2))
        ax.plot([9.3, 9.7], [5.5, 5.5], color="red", lw=2)
        ax.plot([9.5, 9.5], [5.3, 5.7], color="red", lw=2)

    def add_electrochemical_series(ax, x0=11.0, y0=0.0, width=4.5, height=8.3):

        ax.add_patch(Rectangle((x0, y0), width, height,
                            facecolor="white", edgecolor="black", lw=1.5))

        ax.text(x0 + width / 2, y0 + height - 0.35,
                "Electrochemical series",
                ha="center", va="top", fontsize=13, fontweight="bold")

        ax.text(x0 + width - 0.3, y0 + height - 0.85,
                r"$E^\circ_{\mathrm{red}}$ (V)",
                ha="right", va="top", fontsize=10)

        series = sorted(
            STANDARD_REDUCTION_POTENTIALS.items(),
            key=lambda kv: kv[1],
            reverse=True
        )

        y_top = y0 + height - 1.4
        y_bottom = y0 + 2.8
        ys = np.linspace(y_top, y_bottom, len(series))

        for (metal, E), y in zip(series, ys):
            charge = METAL_CHARGES[metal]
            q = format_charge_superscript(charge)
            half_cell = rf"{metal}$^{{{q}}}$ / {metal}"

            is_anode = (metal == anode_metal)
            is_cathode = (metal == cathode_metal)

            if is_anode:
                bg = "#ffe5e5"
                edge = "red"
            elif is_cathode:
                bg = "#e6f0ff"
                edge = "blue"
            else:
                bg = "#f7f7f7"
                edge = "none"

            ax.add_patch(Rectangle((x0 + 0.15, y - 0.23), width - 0.3, 0.46,
                                facecolor=bg, edgecolor=edge,
                                lw=1 if edge != "none" else 0.5))

            ax.text(x0 + 0.3, y, half_cell,
                    ha="left", va="center",
                    fontsize=11,
                    fontweight="bold" if (is_anode or is_cathode) else "normal")

            ax.text(x0 + width - 0.3, y, f"{E:+.2f}",
                    ha="right", va="center", fontsize=10)

        y_cell = y0 + 2.15
        y_react = y0 + 1.55

        if cell_designation is not None:
            ax.text(x0 + 0.2, y_cell,
                    cell_designation,
                    ha="left", va="top", fontsize=12, fontweight="bold")

        if anode_half_reaction is not None:
            ax.text(x0 + 0.2, y_react,
                rf"$\mathbf{{Anode:}}$ {anode_half_reaction}",
                ha="left", va="top", fontsize=10.5)


        if cathode_half_reaction is not None:
            ax.text(x0 + 0.2, y_react - 0.55,
                rf"$\mathbf{{Cathode:}}$ {cathode_half_reaction}",
                ha="left", va="top", fontsize=10.5)

        if cell_potential is not None:
            ax.text(x0 + 0.2, y_react - 1.15,
                rf"$\mathbf{{E^\circ_{{cell}}:}}\ {cell_potential:.2f}\ \mathrm{{V}}$",
                ha="left", va="top", fontsize=10.5)
        
    add_beaker(ax, 0, 0, 4, 4)
    add_beaker(ax, 5, 0, 9, 4)

    add_solution(ax, 0, 4, 3, color="gray", alpha=0.3)
    add_solution(ax, 5, 9, 3, color="tab:blue", alpha=0.2)

    ax.add_patch(Rectangle((0.5, 1), 0.5, 4, facecolor=anode_color, edgecolor="black", lw=2))
    ax.add_patch(Rectangle((8, 1), 0.5, 4, facecolor=cathode_color, edgecolor="black", lw=2))

    y_bottom = 1
    y_top = 3.5

    x_left = 3.0
    x_right = 6.0
    x_center = 0.5 * (x_left + x_right)
    radius = 0.5 * (x_right - x_left)

    ax.plot([x_left, x_left], [y_bottom, y_top], color="black", lw=2)
    ax.add_patch(Arc((x_center, y_top), width=2 * radius, height=2 * radius, theta1=0, theta2=180, lw=2))
    ax.plot([x_right, x_right], [y_top, y_bottom], color="black", lw=2)

    x_left2 = 3.5
    x_right2 = 5.5
    x_center2 = 0.5 * (x_left2 + x_right2)
    radius2 = 0.5 * (x_right2 - x_left2)

    ax.plot([x_left2, x_left2], [y_bottom, y_top], color="black", lw=2)
    ax.add_patch(Arc((x_center2, y_top), width=2 * radius2, height=2 * radius2, theta1=0, theta2=180, lw=2))
    ax.plot([x_right2, x_right2], [y_top, y_bottom], color="black", lw=2)

    wire_x = [0.75, 0.75, 4.5, 8.25, 8.25]
    wire_y = [5.0, 7.0, 7.0, 7.0, 5.0]
    ax.plot(wire_x, wire_y, color="black", lw=2)

    add_voltmeter(ax)

    ax.add_patch(FancyArrowPatch((1.0, 5.75), (2.0, 6.75),
                                 connectionstyle="angle,angleA=90,angleB=0,rad=10",
                                 arrowstyle="->", mutation_scale=14,
                                 lw=2.5, color="red"))

    plus_minus_sign(ax)

    ax.text(4.5, 8.0, title_voltmeter, ha="center", va="center", fontsize=13)
    ax.text(4.5, 5.5, f"{bridge_label} ({salt_bridge_compound})", ha="center", va="center", fontsize=12)

    ax.text(-0.5, 5.0, "Anode", ha="center", va="center", fontsize=12)
    ax.text(-0.5, 4.5, "(oxidation)", ha="center", va="center", fontsize=11)

    ax.text(9.5, 5.0, "Cathode", ha="center", va="center", fontsize=12)
    ax.text(9.5, 4.5, "(reduction)", ha="center", va="center", fontsize=11)

    ax.text(1.5, 6.25, r"$e^-$", color="red", ha="center", va="center", fontsize=13)

    ax.text(2.0, 1.5, anode_electrolyte, ha="center", va="center", fontsize=13)
    ax.text(7.0, 1.5, cathode_electrolyte, ha="center", va="center", fontsize=13)

    ax.text(0.75, 2.0, rf"$\mathbf{{{anode_metal}}}$", ha="center", va="center", fontsize=12)
    ax.text(8.25, 2.0, rf"$\mathbf{{{cathode_metal}}}$", ha="center", va="center", fontsize=12)

    ax.text(1.05, 2.0, anode_reaction, ha="left", va="center", fontsize=10)
    ax.text(7.95, 2.0, cathode_reaction, ha="right", va="center", fontsize=10)

    add_electrochemical_series(ax)

    return fig


# Helper functions

LABEL_WIDTH = "170px"
BOX_WIDTH = "180px"

def format_charge_superscript(charge):
    return "+" if charge == 1 else f"{charge}+"


def electron_count_text(n):
    return r"e$^-$" if n == 1 else rf"{n}e$^-$"

def make_reduction_half_reaction(metal_symbol, charge):
    q = format_charge_superscript(charge)
    e_text = electron_count_text(charge)
    return rf"{metal_symbol}$^{{{q}}}\mathrm{{(aq)}}$ + {e_text} $\rightarrow$ {metal_symbol}(s)"


def update_right_metal_options(*args):
    left_symbol = left_metal_box.value
    left_E = STANDARD_REDUCTION_POTENTIALS[left_symbol]

    valid_right = [
        metal for metal, E in STANDARD_REDUCTION_POTENTIALS.items()
        if metal != left_symbol and E > left_E
    ]

    right_metal_box.options = valid_right

    if right_metal_box.value not in valid_right:
        right_metal_box.value = valid_right[0] if valid_right else None

def make_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=BOX_WIDTH)
    label = widgets.HTML(
        value=f'<span style="font-size:{fontsize}; line-height:1.2;">{label_html}</span>',
        layout=widgets.Layout(width=LABEL_WIDTH)
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))

def show_error(message):
    display(widgets.HTML(
        f"""
        <div style="
            color: #842029;
            background-color: #f8d7da;
            border: 1px solid #f5c2c7;
            border-radius: 6px;
            padding: 10px 12px;
            margin-top: 8px;
            font-weight: 500;
        ">
            {message}
        </div>
        """
    ))


# Widgets

left_metal_box = widgets.Dropdown(options=list(ELECTRODE_LIBRARY.keys()), value="Zn")
right_metal_box = widgets.Dropdown(options=list(ELECTRODE_LIBRARY.keys()), value="Cu")

anode_electrolyte_box = widgets.Dropdown(options=list(ELECTROLYTE_LIBRARY.keys()), value="sulfate")
cathode_electrolyte_box = widgets.Dropdown(options=list(ELECTROLYTE_LIBRARY.keys()), value="sulfate")

salt_bridge_box = widgets.Dropdown(
    options=[
        ("KCl", "KCl"),
        ("Na₂SO₄", "Na_2SO_4"),
    ],
    value="KCl"
)

plot_button = widgets.Button(
    description="Draw cell",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))

left_metal_row = make_row("Anode:", left_metal_box)
right_metal_row = make_row("Cathode:", right_metal_box)
anode_el_row = make_row("Anode electrolyte:", anode_electrolyte_box)
cathode_el_row = make_row("Cathode electrolyte:", cathode_electrolyte_box)
salt_row = make_row("Salt bridge:", salt_bridge_box)
plot_row = widgets.HBox([widgets.HTML(layout=widgets.Layout(width=LABEL_WIDTH)), plot_button])


# When clicked

def on_plot_clicked(b):
    plt.close("all")

    try:
        anode_symbol = left_metal_box.value
        cathode_symbol = right_metal_box.value

        if anode_symbol == cathode_symbol:
            raise ValueError("Anode and cathode should be different metals.")

        anode_electrolyte = ELECTROLYTE_LIBRARY[anode_electrolyte_box.value]
        cathode_electrolyte = ELECTROLYTE_LIBRARY[cathode_electrolyte_box.value]

        anode_charge = METAL_CHARGES[anode_symbol]
        cathode_charge = METAL_CHARGES[cathode_symbol]

        anode_obj = ELECTRODE_LIBRARY[anode_symbol]
        cathode_obj = ELECTRODE_LIBRARY[cathode_symbol]

        anode_salt = build_salt_formula(anode_symbol, anode_charge, anode_electrolyte)
        cathode_salt = build_salt_formula(cathode_symbol, cathode_charge, cathode_electrolyte)

        left_E = STANDARD_REDUCTION_POTENTIALS[anode_symbol]
        right_E = STANDARD_REDUCTION_POTENTIALS[cathode_symbol]

        if left_E >= right_E:
            raise ValueError(
                "Invalid galvanic cell: the Anode must be the more reducing species "
                "(lower standard reduction potential)."
            )

        cell_potential = right_E - left_E
        anode_half_reaction = make_reduction_half_reaction(anode_symbol, anode_charge)
        cathode_half_reaction = make_reduction_half_reaction(cathode_symbol, cathode_charge)
        cell_designation = make_cell_designation(
            anode_symbol, anode_charge, anode_electrolyte,
            cathode_symbol, cathode_charge, cathode_electrolyte
        )

        fig = draw_galvanic_cell(
            anode_metal=anode_symbol,
            cathode_metal=cathode_symbol,
            anode_electrolyte=anode_salt,
            cathode_electrolyte=cathode_salt,
            salt_bridge_compound=SALT_BRIDGE_LIBRARY[salt_bridge_box.value].formula_display,
            anode_reaction=make_anode_reaction(anode_symbol, anode_charge),
            cathode_reaction=make_cathode_reaction(cathode_symbol, cathode_charge),
            anode_half_reaction=anode_half_reaction,
            cathode_half_reaction=cathode_half_reaction,
            cell_potential=cell_potential,
            cell_designation=cell_designation,
            anode_color=anode_obj.color,
            cathode_color=cathode_obj.color,
        )

        out.clear_output(wait=True)
        with out:
            display(fig)
            plt.close(fig)

    except Exception as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Unexpected error: {err}")

plot_button.on_click(on_plot_clicked)


# Overall layout

controls = widgets.VBox(
    [
        left_metal_row,
        right_metal_row,
        anode_el_row,
        cathode_el_row,
        salt_row,
        plot_row,
    ],
    layout=widgets.Layout(width="380px", min_width="380px")
)

output_panel = widgets.Box(
    [out],
    layout=widgets.Layout(flex="1 1 0%", width="auto")
)

ui = widgets.HBox(
    [controls, output_panel],
    layout=widgets.Layout(width="100%", align_items="flex-start")
)

display(ui)